## This notebook shows how we generated the Schubert polynomial dataset. 
## See: https://doc.sagemath.org/html/en/reference/combinat/sage/combinat/schubert_polynomial.html

In [3]:
import numpy as np
import itertools
import random
from sage.all import SchubertPolynomialRing, Permutations, ZZ
import math
random.seed(int(32))

In [4]:
X = SchubertPolynomialRing(ZZ)

In [5]:
#When n = 3, the permutations in the product can all be embedded in S_5
#When n = 4, the permutations in the product can all be embedded in S_7
#When n = 5, the permutations in the product can all be embedded in S_9
#When n = 6, the permutations in the product can all be embedded in S_11

n = 3
nn = 5

In [12]:
def permutation_matrix_from_oneline(p, convention='row'):
    """
    Return the permutation matrix for a 1-line permutation p.

    INPUT:
        - p: list of ints of length n, a permutation of 1..n
             (p[i] = image of i+1)
        - convention: 'row' or 'column'
            * 'row'    : 1s at (i, p[i]-1)     — standard row-action
            * 'column' : 1s at (p[j]-1, j)     — column-action (transpose)

    OUTPUT:
        - an n x n matrix over ZZ with entries in {0,1}
    """
    # basic validation
    n = len(p)
    if sorted(p) != list(range(1, n+1)):
        raise ValueError("p must be a permutation of 1..n")

    M = matrix(ZZ, n, n)  # starts as all zeros

    if convention == 'row':
        # Put a 1 in row i, column p[i]-1
        for i, image in enumerate(p):        # i = 0..n-1, image = p[i] in 1..n
            M[i, image-1] = 1
    elif convention == 'column':
        # Put a 1 in row p[j]-1, column j  (transpose of 'row')
        for j, image in enumerate(p):
            M[image-1, j] = 1
    else:
        raise ValueError("convention must be 'row' or 'column'")

    return M


In [6]:
def swap(perm, ind1, ind2):
    newperm = [0]*len(perm)
    for i in range(len(perm)):
        if i == ind1:
            newperm[i] = perm[ind2]
        elif i == ind2:
            newperm[i] = perm[ind1]
        else:
            newperm[i] = perm[i]
    return newperm

def construct_zero_coeff_example(perm, n):
    #The number of transpositions we multiply perm by is sampled from a geometric distribution
    #The number of transpositions can't be greater than the total number of transpositions
    number_of_transpositions = min( np.random.geometric(0.20), int((nn)*(nn-1)/2))
    combinations = list(itertools.combinations(range( nn ), 2))
    transpositions = random.sample(combinations, number_of_transpositions)
    for (i, j) in transpositions:
        perm = swap(perm, i, j)
    return perm

positive_coeff_triples = []
zero_coeff_triples = []
P = Permutations(n)

for p1 in P:
    for p2 in P:
        #Compute the product of the permutations
        product = X(p1)*X(p2)
        #Make a list of (perm, coeff) that appear in the product
        permutations_in_product = [p[0] for p in list(product)]
        
        #embed permutations in S_{nn}
        embedded_permutations_in_product = [p[0]+ list(range(len(p[0])+1, nn+1 )) for p in list(product)]


        for (perm, coeff) in list(product):
            embedded_perm = perm + list(range(len(perm)+1, nn+1))
            positive_coeff_triples.append((p1, p2, embedded_perm, coeff))

            #Construct an example with a zero coefficient by multiplying the 
            #coefficient in the product by a random number of transpositions
            if len(embedded_perm) > 1:
                newperm = construct_zero_coeff_example(embedded_perm, n)
                
                #Check that the new permutation isn't in the product
                if newperm not in embedded_permutations_in_product: 
                    zero_coeff_triples.append((p1, p2, newperm, 0))
                else:
                    print(f"{newperm} in {embedded_permutations_in_product}, not adding to zero coeff triples")
                    
                    
                    


In [7]:
len(positive_coeff_triples)

43

In [8]:
len(zero_coeff_triples)

43

In [9]:
all_examples = positive_coeff_triples + zero_coeff_triples

In [10]:
random.shuffle(all_examples)
split = 0.8
ds_size = int(len(all_examples))

all_examples_train = all_examples[:math.ceil(ds_size*split)]
all_examples_test = all_examples[math.ceil(ds_size*split):]


In [11]:
arr_train = []
for row in all_examples_train:
    arr_train.append(str(row))
arr_test = []
for row in all_examples_test:
    arr_test.append(str(row))
np.savetxt(f'schubert_{n}_train.txt', arr_train, fmt = "%s")
np.savetxt(f'schubert_{n}_test.txt', arr_test, fmt = "%s")

In [15]:
# If you already defined this earlier, you can skip redefining it.
def permutation_matrix_from_oneline(p, convention='row'):
    n = len(p)
    if sorted(p) != list(range(1, n+1)):
        raise ValueError(f"Not a valid permutation of 1..{n}: {p}")
    M = matrix(ZZ, n, n)
    if convention == 'row':
        for i, image in enumerate(p):
            M[i, image-1] = 1
    elif convention == 'column':
        for j, image in enumerate(p):
            M[image-1, j] = 1
    else:
        raise ValueError("convention must be 'row' or 'column'")
    return M

# --- Converters for your dataset ---

def _to_perm_matrix(x, *, convention='row'):
    """
    Accepts a list/tuple (like [1,3,2]) or a Sage matrix (passes through),
    and returns a Sage permutation matrix.
    """
    # Pass through if it's already a matrix-like object
    if hasattr(x, 'nrows') and hasattr(x, 'ncols'):
        return x
    return permutation_matrix_from_oneline(list(x), convention=convention)

def convert_example(example, *, convention='row'):
    """
    Converts one example tuple:
       ([..], [..], [..], label)  ->  (M1, M2, M3, label)
    """
    a, b, c, label = example
    return (_to_perm_matrix(a, convention=convention),
            _to_perm_matrix(b, convention=convention),
            _to_perm_matrix(c, convention=convention),
            label)

# Convert the whole training list
all_examples_train_mats = [convert_example(ex, convention='row') for ex in all_examples_train]

# (Optional) Do the same for your test/val lists if you have them:
# all_examples_val_mats   = [convert_example(ex) for ex in all_examples_val]
# all_examples_test_mats  = [convert_example(ex) for ex in all_examples_test]


In [16]:
for idx, ex in enumerate(all_examples_train):
    try:
        convert_example(ex)  # will raise if something's off
    except Exception as e:
        print(f"Problem at index {idx}: {e}")
        break
else:
    print("All examples converted successfully ✅")


All examples converted successfully ✅


In [17]:
all_examples_train_mats

[(
                  [0 0 0 1 0]   
                  [1 0 0 0 0]   
[0 0 1]  [0 1 0]  [0 1 0 0 0]   
[1 0 0]  [1 0 0]  [0 0 1 0 0]   
[0 1 0], [0 0 1], [0 0 0 0 1], 1
),
 (
                  [1 0 0 0 0]   
                  [0 0 0 1 0]   
[1 0 0]  [1 0 0]  [0 1 0 0 0]   
[0 0 1]  [0 0 1]  [0 0 1 0 0]   
[0 1 0], [0 1 0], [0 0 0 0 1], 1
),
 (
                  [1 0 0 0 0]   
                  [0 1 0 0 0]   
[1 0 0]  [0 0 1]  [0 0 0 0 1]   
[0 1 0]  [0 1 0]  [0 0 0 1 0]   
[0 0 1], [1 0 0], [0 0 1 0 0], 0
),
 (
                  [0 1 0 0 0]   
                  [1 0 0 0 0]   
[1 0 0]  [0 1 0]  [0 0 1 0 0]   
[0 1 0]  [1 0 0]  [0 0 0 1 0]   
[0 0 1], [0 0 1], [0 0 0 0 1], 1
),
 (
                  [0 0 1 0 0]   
                  [1 0 0 0 0]   
[0 0 1]  [1 0 0]  [0 1 0 0 0]   
[1 0 0]  [0 1 0]  [0 0 0 1 0]   
[0 1 0], [0 0 1], [0 0 0 0 1], 1
),
 (
                  [0 0 0 0 1]   
                  [0 0 1 0 0]   
[0 0 1]  [1 0 0]  [0 1 0 0 0]   
[1 0 0]  [0 0 1]  [1 0 0 0 0]   
[0 1 0], [

In [18]:
all_examples_train

[([3, 1, 2], [2, 1, 3], [4, 1, 2, 3, 5], 1),
 ([1, 3, 2], [1, 3, 2], [1, 4, 2, 3, 5], 1),
 ([1, 2, 3], [3, 2, 1], [1, 2, 5, 4, 3], 0),
 ([1, 2, 3], [2, 1, 3], [2, 1, 3, 4, 5], 1),
 ([3, 1, 2], [1, 2, 3], [3, 1, 2, 4, 5], 1),
 ([3, 1, 2], [1, 3, 2], [5, 3, 2, 1, 4], 0),
 ([3, 1, 2], [2, 1, 3], [1, 2, 4, 3, 5], 0),
 ([1, 2, 3], [2, 1, 3], [3, 1, 5, 4, 2], 0),
 ([1, 2, 3], [1, 2, 3], [4, 1, 2, 3, 5], 0),
 ([3, 1, 2], [3, 2, 1], [1, 4, 2, 3, 5], 0),
 ([1, 3, 2], [3, 1, 2], [4, 1, 2, 3, 5], 1),
 ([3, 1, 2], [1, 2, 3], [1, 2, 3, 4, 5], 0),
 ([2, 1, 3], [2, 3, 1], [4, 3, 5, 2, 1], 0),
 ([1, 3, 2], [2, 1, 3], [3, 1, 2, 4, 5], 1),
 ([3, 2, 1], [1, 3, 2], [3, 4, 1, 2, 5], 1),
 ([1, 3, 2], [1, 2, 3], [1, 3, 2, 4, 5], 1),
 ([2, 1, 3], [3, 2, 1], [5, 3, 1, 4, 2], 0),
 ([2, 1, 3], [2, 3, 1], [3, 2, 1, 4, 5], 1),
 ([1, 3, 2], [1, 2, 3], [3, 2, 1, 5, 4], 0),
 ([2, 1, 3], [2, 1, 3], [5, 4, 2, 3, 1], 0),
 ([2, 1, 3], [3, 1, 2], [4, 1, 2, 3, 5], 1),
 ([3, 2, 1], [1, 2, 3], [2, 3, 4, 1, 5], 0),
 ([3, 2, 1